In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import gc

In [5]:
# TRAIN 2017
train_csv_path = "//content/drive/MyDrive/Colab Notebooks/modele stocastique/full_combined_df_final.csv"

# TEST 2016 Q4
test_csv_path = "/content/drive/MyDrive/Colab Notebooks/modele stocastique/df_q4_test.csv"

# DOSSIER SAVE
save_path = "/content/drive/MyDrive/Colab Notebooks/modele stocastique/"

In [6]:
# =========================================================
# CELLULE 4 : Filtrer seulement le modèle ST4000DM000
# Utilisation de chunks pour économiser la mémoire
# =========================================================

model_name = "ST4000DM000"

train_chunks = []

for chunk in pd.read_csv(
    train_csv_path,
    chunksize=100000,
    low_memory=False
):

    filtered_chunk = chunk[chunk["model"] == model_name]

    if len(filtered_chunk) > 0:
        train_chunks.append(filtered_chunk)

train_df = pd.concat(train_chunks, ignore_index=True)

print("Train shape :", train_df.shape)
train_df.head()

Train shape : (9085357, 95)


,date,serial_number,model,capacity_bytes,failure,smart_1_normalized,smart_1_raw,smart_2_normalized,smart_2_raw,smart_3_normalized,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2017-02-05,Z305B2QN,ST4000DM000,4000787030016,0,109.0,22966656.0,NaN,NaN,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-02-05,Z302A0YH,ST4000DM000,4000787030016,0,117.0,129645632.0,NaN,NaN,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-02-05,Z305BT0W,ST4000DM000,4000787030016,0,120.0,237273656.0,NaN,NaN,93.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-02-05,Z302A0YE,ST4000DM000,4000787030016,0,117.0,154434344.0,NaN,NaN,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-02-05,Z302PGH8,ST4000DM000,4000787030016,0,110.0,28993712.0,NaN,NaN,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# =========================================================
# CELLULE 5 : Filtrer le TEST
# =========================================================

test_chunks = []

for chunk in pd.read_csv(
    test_csv_path,
    chunksize=100000,
    low_memory=False
):

    filtered_chunk = chunk[chunk["model"] == model_name]

    if len(filtered_chunk) > 0:
        test_chunks.append(filtered_chunk)

test_df = pd.concat(test_chunks, ignore_index=True)

print("Test shape :", test_df.shape)
test_df.head()

Test shape : (3196552, 95)


,date,serial_number,model,capacity_bytes,failure,smart_1_normalized,smart_1_raw,smart_2_normalized,smart_2_raw,smart_3_normalized,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2016-10-31,Z305B2QN,ST4000DM000,4000787030016,0,102.0,4986112.0,NaN,NaN,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-10-31,Z302A0YH,ST4000DM000,4000787030016,0,114.0,77504120.0,NaN,NaN,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-10-31,Z305BT0W,ST4000DM000,4000787030016,0,117.0,123542688.0,NaN,NaN,93.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-10-31,Z302A0YE,ST4000DM000,4000787030016,0,110.0,26166272.0,NaN,NaN,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-10-31,Z302PGH8,ST4000DM000,4000787030016,0,118.0,177250904.0,NaN,NaN,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# =========================================================
# CELLULE 6 : Vérification des dimensions
# =========================================================

print("TRAIN")
print(train_df.shape)

print("\nTEST")
print(test_df.shape)

TRAIN
(9085357, 95)

TEST
(3196552, 95)


In [9]:
# =========================================================
# CELLULE 7 : Supprimer les colonnes totalement vides
# (comme demandé dans le PDF)
# =========================================================

train_df = train_df.dropna(axis=1, how="all")
test_df = test_df.dropna(axis=1, how="all")

print("Train shape :", train_df.shape)
print("Test shape  :", test_df.shape)

Train shape : (9085357, 53)
Test shape  : (3196552, 53)


In [10]:
# =========================================================
# CELLULE 8 : Détection des colonnes constantes
# seulement sur les colonnes SMART
# =========================================================

smart_columns = [col for col in train_df.columns if "smart" in col]

constant_columns = []

for col in smart_columns:

    try:
        if train_df[col].nunique(dropna=False) == 1:
            constant_columns.append(col)
    except:
        pass

print("Nombre de colonnes constantes :", len(constant_columns))
print(constant_columns)

Nombre de colonnes constantes : 0
[]


In [11]:
# =========================================================
# CELLULE 9 : Supprimer les colonnes constantes
# =========================================================

train_df = train_df.drop(columns=constant_columns, errors="ignore")
test_df = test_df.drop(columns=constant_columns, errors="ignore")

print("Train shape :", train_df.shape)
print("Test shape  :", test_df.shape)

Train shape : (9085357, 53)
Test shape  : (3196552, 53)


In [12]:
# =========================================================
# CELLULE 10 : Identifier les colonnes avec valeurs manquantes
# =========================================================

missing_train = train_df.isnull().sum()
missing_train = missing_train[missing_train > 0]

print(missing_train.sort_values(ascending=False))

smart_1_normalized      200
smart_1_raw             200
smart_3_normalized      200
smart_3_raw             200
smart_4_normalized      200
smart_4_raw             200
smart_5_normalized      200
smart_5_raw             200
smart_7_normalized      200
smart_7_raw             200
smart_9_normalized      200
smart_9_raw             200
smart_10_normalized     200
smart_10_raw            200
smart_12_normalized     200
smart_12_raw            200
smart_183_normalized    200
smart_183_raw           200
smart_184_normalized    200
smart_184_raw           200
smart_187_normalized    200
smart_187_raw           200
smart_188_normalized    200
smart_188_raw           200
smart_189_normalized    200
smart_189_raw           200
smart_190_normalized    200
smart_190_raw           200
smart_191_normalized    200
smart_191_raw           200
smart_192_normalized    200
smart_192_raw           200
smart_193_normalized    200
smart_193_raw           200
smart_194_normalized    200
smart_194_raw       

In [13]:
# =========================================================
# CELLULE 11 : Remplacer les valeurs manquantes par 0
# (comme proposé dans le PDF)
# =========================================================

train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

print("Valeurs manquantes TRAIN :", train_df.isnull().sum().sum())
print("Valeurs manquantes TEST  :", test_df.isnull().sum().sum())

Valeurs manquantes TRAIN : 0
Valeurs manquantes TEST  : 0


In [14]:
# =========================================================
# CELLULE 12 : Vérifier que les colonnes train/test sont identiques
# =========================================================

common_columns = train_df.columns.intersection(test_df.columns)

train_df = train_df[common_columns]
test_df = test_df[common_columns]

print(len(common_columns))

53


In [15]:
# =========================================================
# CELLULE 13 : Réduction mémoire
# =========================================================

for col in train_df.select_dtypes(include=["float64"]).columns:
    train_df[col] = train_df[col].astype("float32")

for col in train_df.select_dtypes(include=["int64"]).columns:
    train_df[col] = train_df[col].astype("int32")

for col in test_df.select_dtypes(include=["float64"]).columns:
    test_df[col] = test_df[col].astype("float32")

for col in test_df.select_dtypes(include=["int64"]).columns:
    test_df[col] = test_df[col].astype("int32")

print("Mémoire optimisée")

Mémoire optimisée


In [16]:
# =========================================================
# CELLULE 14 : Identifier les colonnes contenant "normalized"
# =========================================================

normalized_columns = [
    col for col in train_df.columns
    if "normalized" in col.lower()
]

print("Colonnes normalized :")
print(normalized_columns)
print("Nombre :", len(normalized_columns))

Colonnes normalized :
['smart_1_normalized', 'smart_3_normalized', 'smart_4_normalized', 'smart_5_normalized', 'smart_7_normalized', 'smart_9_normalized', 'smart_10_normalized', 'smart_12_normalized', 'smart_183_normalized', 'smart_184_normalized', 'smart_187_normalized', 'smart_188_normalized', 'smart_189_normalized', 'smart_190_normalized', 'smart_191_normalized', 'smart_192_normalized', 'smart_193_normalized', 'smart_194_normalized', 'smart_197_normalized', 'smart_198_normalized', 'smart_199_normalized', 'smart_240_normalized', 'smart_241_normalized', 'smart_242_normalized']
Nombre : 24


In [17]:
# =========================================================
# CELLULE 15 : Supprimer les colonnes normalized
# =========================================================

train_df = train_df.drop(columns=normalized_columns, errors="ignore")
test_df = test_df.drop(columns=normalized_columns, errors="ignore")

print("Colonnes normalized supprimées")

Colonnes normalized supprimées


In [18]:
# =========================================================
# CELLULE 16 : Supprimer model et capacity_bytes
# =========================================================

columns_to_remove = ["model", "capacity_bytes"]

train_df = train_df.drop(columns=columns_to_remove, errors="ignore")
test_df = test_df.drop(columns=columns_to_remove, errors="ignore")

print("Colonnes supprimées")

Colonnes supprimées


In [19]:
# =========================================================
# CELLULE 17 : Convertir date en datetime
# =========================================================

train_df["date"] = pd.to_datetime(train_df["date"])
test_df["date"] = pd.to_datetime(test_df["date"])

print("Conversion terminée")

Conversion terminée


In [20]:
# =========================================================
# CELLULE 18 : Vérifier le type de la colonne date
# =========================================================

print(train_df["date"].dtype)
print(test_df["date"].dtype)

datetime64[ns]
datetime64[ns]


In [21]:
# =========================================================
# CELLULE 19 : Trier les données
# =========================================================

train_df = train_df.sort_values(
    by=["serial_number", "date"]
)

test_df = test_df.sort_values(
    by=["serial_number", "date"]
)

print("Tri terminé")

Tri terminé


In [22]:
# =========================================================
# CELLULE 20 : Vérifier le tri chronologique
# =========================================================

sample_disk = train_df["serial_number"].iloc[0]

train_df[
    train_df["serial_number"] == sample_disk
][["serial_number", "date"]].head(20)

,serial_number,date
3544102,S3000A9T,2017-06-03
5989621,S3000A9T,2017-06-04
5507368,S3000A9T,2017-06-05
4818455,S3000A9T,2017-06-06
4164079,S3000A9T,2017-06-07
5438476,S3000A9T,2017-06-08
3923065,S3000A9T,2017-06-09
3819705,S3000A9T,2017-06-10
3716341,S3000A9T,2017-06-11
4852863,S3000A9T,2017-06-12


In [23]:
# =========================================================
# CELLULE 21 : Réinitialiser les index
# =========================================================

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Index réinitialisés")

Index réinitialisés


In [24]:
# =========================================================
# CELLULE 22 : Vérifier les valeurs manquantes restantes
# =========================================================

print("NaN TRAIN :", train_df.isnull().sum().sum())
print("NaN TEST  :", test_df.isnull().sum().sum())

NaN TRAIN : 0
NaN TEST  : 0


In [25]:
# =========================================================
# CELLULE 23 : Sauvegarder les nouveaux CSV
# =========================================================

train_csv_filtered = save_path + "train_ST4000DM000_cleaned.csv"
test_csv_filtered = save_path + "test_ST4000DM000_cleaned.csv"

train_df.to_csv(train_csv_filtered, index=False)
test_df.to_csv(test_csv_filtered, index=False)

print("CSV sauvegardés")

CSV sauvegardés


In [26]:
# =========================================================
# CELLULE 24 : Sauvegarder en PKL
# =========================================================

train_pkl_path = save_path + "train_ST4000DM000_cleaned.pkl"
test_pkl_path = save_path + "test_ST4000DM000_cleaned.pkl"

train_df.to_pickle(train_pkl_path)
test_df.to_pickle(test_pkl_path)

print("PKL sauvegardés")

PKL sauvegardés


In [27]:
# =========================================================
# CELLULE 25 : Vérification du chargement PKL
# =========================================================

train_check = pd.read_pickle(train_pkl_path)
test_check = pd.read_pickle(test_pkl_path)

print("Train loaded :", train_check.shape)
print("Test loaded  :", test_check.shape)

Train loaded : (9085357, 27)
Test loaded  : (3196552, 27)
